In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount. Safe for a brand-new Runtime → Run all.
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/LCX_Curved_Template_Reacquisition_v1_cacheonly_masterpath_fixed'
BRANCH='lcx-curved-template-reacquisition-from-main'


# OpenPlaque — LCX curved-template reacquisition — Master-path fixed

Same scientific experiment, with two runtime corrections: (1) cached curved RCA/LCX NIfTIs are used instead of the missing historical `Full_DICOM.zip`; and (2) Master Coronary Anatomy Baseline v2 is loaded from its actual location under `OpenPlaque/Cache/`. The stale historical LCX source centerline remains forbidden.


In [ ]:
import shutil, subprocess, sys
repo='/content/OpenPlaque'
shutil.rmtree(repo, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo,'pytest','scikit-image'], check=True)
print('COMMIT:')
subprocess.run(['git','-C',repo,'rev-parse','HEAD'], check=True)
print('IMPORT CHECK:')
subprocess.run([sys.executable,'-c','import openplaque.lcx_curved_template_reacquisition_v3 as m; print(m.ALGORITHM); print(m.MASTER)'], check=True)
print('SYNTAX CHECK:')
subprocess.run([sys.executable,'-m','py_compile',repo + '/src/openplaque/lcx_curved_template_reacquisition.py',repo + '/src/openplaque/lcx_curved_template_reacquisition_v2.py',repo + '/src/openplaque/lcx_curved_template_reacquisition_v3.py'], check=True)
print('TESTS:')
subprocess.run([sys.executable,'-m','pytest','-q',repo + '/tests/test_lcx_curved_template_reacquisition.py',repo + '/tests/test_lcx_curved_template_reacquisition_v2.py',repo + '/tests/test_lcx_curved_template_reacquisition_v3.py'], check=True)


In [ ]:
# Explicit preflight against the exact inputs used by v3.
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/total/aorta.nii.gz',
 root/'UCLA_Plaque_Context_Verification/RCA_input/RCA_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/LCX_input/LCX_0000.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/RCA.nii.gz',
 root/'UCLA_Plaque_Context_Verification/nnunet_masks/LCX.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print(f'INPUT PREFLIGHT: {len(required)-len(missing)}/{len(required)} present')
for p in required:
    print(('OK   ' if p.exists() else 'MISS '), p)
if missing:
    raise FileNotFoundError('Missing required cached inputs:\n'+'\n'.join(missing))


In [ ]:
# Run the corrected cache-only workflow in a fresh Python subprocess.
import subprocess, sys, textwrap
runner=textwrap.dedent(f'''
from openplaque.lcx_curved_template_reacquisition_v3 import synthetic_lcx_template_v3_self_test, run
print('SELF TEST:', synthetic_lcx_template_v3_self_test(), flush=True)
result=run({DRIVE_ROOT!r}, {OUTPUT_ROOT!r})
print('STATUS:', result['summary']['status'], flush=True)
print('REPORT:', result['report'], flush=True)
print('ZIP:', result['zip'], flush=True)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LCX_CURVED_TEMPLATE_REACQUISITION_REPORT_BACK.zip', flush=True)
''')
proc=subprocess.run([sys.executable,'-c',runner], text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print('--- WORKFLOW STDERR ---')
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'LCX workflow failed with exit code {proc.returncode}; full traceback is printed above')
